# Fine-tuning:  — Final Evaluation (Dev Set)

Цель ноутбука — Провести **честное сравнение стратегий обучения** при фиксированных гиперпараметрах и одинаковых условиях.

Сравниваются:

- Scratch (обучение с нуля)
- Full fine-tuning (full_ft)
- Low LR encoder (low_lr_encoder)
- Partial fine-tuning (partial_ft)
- Warmup fine-tuning (warmup)

---

## ⚙️ Фиксированные условия

Все эксперименты проводятся при **одинаковых настройках**:

- `lr_encoder = 3e-5`
- `lr_head = 3e-4`
- `weight_decay = 1e-3`
- `warmup_epochs = 3`

Важно:
- ❗ **гиперпараметры НЕ подбираются**
- ❗ **split’ы фиксированы**
- ❗ **нормализация считается только по Calib_p субъекта**
- ❗ **test_rest не используется при обучении**

---

## 🧪 Протокол

Для каждого субъекта и каждого уровня калибровки:

\[
p \in \{10, 20, 40, 60, 100\}
\]

выполняется:

1. Формирование Calib_p (вложенные подвыборки)
2. Train/Val split внутри Calib_p (stratified)
3. Обучение модели
4. Early stopping по `val_loss`
5. Оценка на фиксированном `test_rest`

---

## 📊 Метрики

Для каждой конфигурации сохраняются:

- ROC-AUC (основная метрика)
- Accuracy
- Binary F1
- Precision / Recall
- FDR

---

## 🧠 Архитектура модели

- Encoder: `UNet1DEncoder`
- Head: Global Average Pooling (по времени) + Linear (512 → 2)
- Loss: CrossEntropy

---

## 🔄 Использование SSL

Рассматриваются два режима:

- **Scratch** — случайная инициализация encoder
- **SSL** — encoder предобучен (masked reconstruction на BigP3BCI)

---

## ⚠️ Важно про Dev-выборку

Этот ноутбук используется для:

- проверки стабильности pipeline
- sanity-check результатов
- анализа поведения стратегий

❗ Результаты на Dev **НЕ являются финальными**

Финальные выводы для диплома будут сделаны на **Test-выборке**, которая полностью отделена и не использовалась на этапе выбора стратегий и гиперпараметров.

---

## 📦 Ожидаемый результат

- Таблица: `p × strategy × metric`
- Графики:
  - ROC-AUC vs p
  - F1 vs p

---

## 🚀 Следующий шаг

После завершения данного этапа:

👉 аналогичный эксперимент будет проведён на **Test-выборке**  
👉 результаты Test будут использованы в дипломе как финальные

---


## 1. Импорты

In [ ]:
import os
import json
import math
import random
import gc
from pathlib import Path
from copy import deepcopy
from itertools import product

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

import stage5_utils as u

import model_unet as m


## 2. Общие конфиги

In [ ]:
RUNTIME_CONFIG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "pin_memory": torch.cuda.is_available(),

    "batch_size": 64,
    "num_workers": 2,

    "max_epochs": 100,
    "patience": 10,
    "min_delta": 0.0,
    "fallback_p_for_zero": 10,
    "val_ratio": 0.2,
}

In [ ]:
PATHS = {
    "data_root": Path("/kaggle/input/datasets/taisiyaglazova"),
    "encoder_checkpoint": Path("/kaggle/input/datasets/taisiyaglazova/ssl-full-encoder-best/encoder_best.pt"),
    "results_root": Path("/kaggle/working/stage5_results"),
}

## 3. Воспроизводимость

In [ ]:
u.set_seed(RUNTIME_CONFIG["seed"])
print("Device:", RUNTIME_CONFIG["device"])

## 4. Пути

In [ ]:
# Пути к датасетам
DATASETS = {
    "bigp3_train": PATHS["data_root"] / "bigp3bci-downstream-train",
    "bigp3_benchmark": PATHS["data_root"] / "bigp3bci-downstream-benchmark",
    "bcicomp3": PATHS["data_root"] / "bcicompiii-ds2",  
}

In [ ]:
GROUPS = ["train", "benchmark", "bcicomp3"]

### Автореестр субъектов

In [ ]:
def list_subject_ids(group: str):
    assert group in GROUPS, f"Unknown group: {group}"

    if group == "train":
        data_dir = DATASETS["bigp3_train"] / "train"
    elif group == "benchmark":
        data_dir = DATASETS["bigp3_benchmark"]/ "benchmark"

    subject_ids = sorted([p.stem for p in data_dir.glob("*.npz")])
    return subject_ids

In [ ]:
# Сборка реестра
SUBJECT_REGISTRY = {
    "train": list_subject_ids("train"),
    "benchmark": list_subject_ids("benchmark"),
}

print("Train subjects:", len(SUBJECT_REGISTRY["train"]))
print("Benchmark subjects:", len(SUBJECT_REGISTRY["benchmark"]))
print("Example train:", SUBJECT_REGISTRY["train"][:5])
print("Example benchmark:", SUBJECT_REGISTRY["benchmark"][:5])


## 5. Конфиг для full Dev run

In [ ]:
FINAL_DEV_CONFIG = {
    "tag": "stage5_final_eval_dev_ft_strategies",
    "subjects": SUBJECT_REGISTRY["train"],
    "group": "train",

    "p_list": [10, 20, 40, 60, 100],
    "scenarios": ["full_ft", "low_lr_encoder", "partial_ft", "warmup"],
    "seed_list": [42],

    "lr_encoder": 3e-5,
    "lr_head": 3e-4,
    "weight_decay": 1e-3,
    "warmup_epochs": 3,
}

RUN_DIR = PATHS["results_root"] / FINAL_DEV_CONFIG["tag"]

ARTIFACT_DIRS = {
    "root": RUN_DIR,
    "history": RUN_DIR / "history",
    "predictions": RUN_DIR / "predictions",
    "tables": RUN_DIR / "tables",
    "figures": RUN_DIR / "figures",
}

for path in ARTIFACT_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print("RUN_DIR:", RUN_DIR)

## Полный запуск Full Dev-Fine-tuning

In [ ]:
final_dev_df = run_many_block4(
    subject_list=FINAL_DEV_CONFIG["subjects"],
    p_list=FINAL_DEV_CONFIG["p_list"],
    ft_strategy_list=FINAL_DEV_CONFIG["ft_strategy_list"],
    seed_list=FINAL_DEV_CONFIG["seed_list"],
    group=FINAL_DEV_CONFIG["group"],
    results_root=FINAL_DEV_DIR,
    encoder_checkpoint=CONFIG["encoder_checkpoint"],
    scenario="ssl_ft",
    lr_encoder=FINAL_DEV_CONFIG["lr_encoder"],
    lr_head=FINAL_DEV_CONFIG["lr_head"],
    weight_decay=FINAL_DEV_CONFIG["weight_decay"],
    warmup_epochs=FINAL_DEV_CONFIG["warmup_epochs"],
    save_history=True,
    save_predictions=True,
    save_summary_every=1,
    continue_on_error=True,
)

In [ ]:
final_dev_df = u.run_many(
    subject_list=FINAL_DEV_CONFIG["subjects"],
    p_list=FINAL_DEV_CONFIG["p_list"],
    scenario_list=["ssl_ft"],
    group=FINAL_DEV_CONFIG["group"],
    runtime_config=RUNTIME_CONFIG,
    results_root=RUN_DIR,
    encoder_checkpoint=PATHS["encoder_checkpoint"],
    ft_strategy_list=FINAL_DEV_CONFIG["scenarios"],
    lr_encoder=FINAL_DEV_CONFIG["lr_encoder"],
    lr_head=FINAL_DEV_CONFIG["lr_head"],
    weight_decay=FINAL_DEV_CONFIG["weight_decay"],
    warmup_epochs=FINAL_DEV_CONFIG["warmup_epochs"],
    save_history=True,
    save_predictions=True,
    save_summary_every=1,
    continue_on_error=True,
)

In [ ]:
print(type(final_dev_df))
print(final_dev_df.shape if hasattr(final_dev_df, "shape") else "no shape")
print(final_dev_df.columns if hasattr(final_dev_df, "columns") else "no columns")
display(final_dev_df)